In [ ]:
!pip install diffusers==0.30.0 transformers accelerate safetensors
!pip install torch --index-url https://download.pytorch.org/whl/cu121

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.8 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.35.2
    Uninstalling diffusers-0.35.2:
      Successfully uninstalled diffusers-0.35.2
Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image
import torch
import os

# ==========================
# 設定區
# ==========================
INPUT_DIR = "/content/drive/MyDrive/test_P/"     # 原圖資料夾
MASK_DIR  = "/content/drive/MyDrive/mask_outputs"      # mask 資料夾
OUT_DIR   = "./output_inpaint/"    # 輸出資料夾
os.makedirs(OUT_DIR, exist_ok=True)

PROMPT = "clean seamless background, smooth continuous texture, natural color matching, realistic background reconstruction, high-quality inpainting, photo-realistic detail"
NEG_PROMPT = "watermark, words, text, blur, artifacts, logo, patchy texture, repeating patterns, rough edges, distorted shapes, color mismatch, noisy pixels, low-quality restoration"

STEPS = 100
GUIDE = 7.0
# ==========================

# --------------------------
# 載入 Inpainting 模型
# --------------------------
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16
).to("cuda")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

safety_checker/model.safetensors not found


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.bin:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

vae/diffusion_pytorch_model.bin:   0%|          | 0.00/335M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


In [ ]:
# --------------------------
# 批次處理資料夾影像
# --------------------------
image_files = sorted([
    f for f in os.listdir(INPUT_DIR)
    if f.lower().endswith((".png", ".jpg", ".jpeg"))
])

print(f"共找到 {len(image_files)} 張圖片\n")

for fname in image_files:
    img_path = os.path.join(INPUT_DIR, fname)
    mask_path = os.path.join(MASK_DIR, fname[:-4]+".png")

    if not os.path.exists(mask_path):
        print(f"[Skip] 找不到 mask：{fname}")
        continue

    # 讀圖
    init_image = Image.open(img_path).convert("RGB")
    mask_image = Image.open(mask_path).convert("RGB")

    # Inpainting
    result = pipe(
        prompt=PROMPT,
        negative_prompt=NEG_PROMPT,
        image=init_image,
        mask_image=mask_image,
        num_inference_steps=STEPS,
        guidance_scale=GUIDE,
    ).images[0]

    # 存檔
    out_path = os.path.join(OUT_DIR, fname)
    result.save(out_path)

    print(f"[OK] {fname}")

print("\n全部完成！")


共找到 1395 張圖片



  0%|          | 0/100 [00:00<?, ?it/s]

[OK] 00C8DD.jpg


  0%|          | 0/100 [00:00<?, ?it/s]

[OK] 0498XA.jpg


...

[OK] F6HAAF.jpg


  0%|          | 0/100 [00:00<?, ?it/s]

[OK] F777UB.jpg


  0%|          | 0/100 [00:00<?, ?it/s]

[OK] F87MOF.jpg


In [ ]:
!zip -r inpaint.zip output_inpaint
from google.colab import files
files.download('mydata.zip')